# 01 - Treat Star Ratings

## Purpose
Combine all 12 quarterly Star Ratings snapshots (May 2023 â†’ Feb 2026) into a single clean file,
then enrich with SA3 geographic codes by joining to the service list.

## Input
- `data/raw/star_ratings/` â€” 12 XLSX files, one per quarterly release

## Output
- `data/clean/star_ratings_by_facility.csv` â€” facility Ã— snapshot, with quality scores and SA3 codes

## Used in
- **Chapter 1** (trend line: did the Oct 2023 staffing mandate improve quality?)
- **Chapter 4** (SA3-level quality score for the master join)

## Key context
- Star Ratings launched **Dec 2022** â€” all data here is post-launch
- **Oct 2023** staffing mandate: 400 care mins/day, 40 RN mins/day â€” major turning point
- Quality score = mean of 4 sub-dimensions: Residents' Experience, Staffing, Compliance, Quality Measures
- MMM Code: MM1=Metro â†’ MM7=Very Remote (7 remoteness classes)

In [25]:
import pandas as pd
import os

RAW_STARS   = '../../data/raw/star_ratings'
RAW_SERVICE = '../../data/raw/service_list'
OUT         = '../../data/clean/star_ratings_by_facility.csv'

# Month name â†’ number mapping for parsing snapshot dates
MONTH_MAP = {
    'January':1, 'February':2, 'March':3, 'April':4,
    'May':5, 'June':6, 'July':7, 'August':8,
    'September':9, 'October':10, 'November':11, 'December':12
}

In [26]:
# =============================================================================
# STEP 1: Load and combine all 12 star ratings files
# =============================================================================
# Each file covers one quarterly snapshot. We prefer the "Detailed data" sheet
# (superset of "Star Ratings" — includes care minutes actuals/targets).
# Older files (pre-mandate: May/Aug 2023) may only have "Star Ratings" sheet —
# the fallback handles this; care minutes columns will be NaN for those rows.

frames = []
for fname in sorted(os.listdir(RAW_STARS)):
    if not fname.endswith('.xlsx') or fname.startswith('~$'):
        continue
    path   = f'{RAW_STARS}/{fname}'
    xl     = pd.ExcelFile(path)
    sheets = xl.sheet_names
    if 'Detailed data' in sheets:
        sheet = 'Detailed data'
    else:
        sheet = [s for s in sheets if 'Star' in s or 'star' in s][0]
    df = pd.read_excel(path, sheet_name=sheet)
    df['source_file'] = fname
    frames.append(df)

stars = pd.concat(frames, ignore_index=True)
print(f'Combined shape: {stars.shape}')
print(f'Snapshots found: {stars["Reporting Period"].unique()}')

Combined shape: (31290, 80)
Snapshots found: <ArrowStringArray>
[  'August 2023',   'August 2025', 'December 2023', 'February 2024',
  'January 2025', 'February 2026',     'July 2024',      'May 2023',
      'May 2024',      'May 2025', 'November 2024',  'October 2025']
Length: 12, dtype: str


In [27]:
# =============================================================================
# STEP 2: Standardise column names and fix data issues
# =============================================================================

stars = stars.rename(columns={
    'Reporting Period'                          : 'snapshot',
    'State/Territory'                           : 'state',
    'Aged Care Planning Region'                 : 'acpr_name',
    'MMM Code'                                  : 'mmm_code',
    'MMM Region'                                : 'mmm_region',
    'Overall Star Rating'                       : 'overall_rating',
    "Residents' Experience rating"              : 'residents_exp',
    'Staffing rating'                           : 'staffing',
    'Compliance rating'                         : 'compliance',
    'Quality Measures rating'                   : 'quality_measures',
    '[S] Registered Nurse Care Minutes - Target': 'rn_minutes_target',
    '[S] Registered Nurse Care Minutes - Actual': 'rn_minutes_actual',
    '[S] Total Care Minutes - Target'           : 'total_minutes_target',
    '[S] Total Care Minutes - Actual'           : 'total_minutes_actual',
})

# Known data issue: the February 2025 file has 'January 2025' in Reporting Period.
stars['snapshot'] = stars.apply(
    lambda r: 'February 2025'
    if r['snapshot'] == 'January 2025' and 'february-2025' in str(r['source_file'])
    else r['snapshot'],
    axis=1
)

# Convert rating columns to numeric (some cells contain suppression markers like 'np')
rating_cols = ['residents_exp', 'staffing', 'compliance', 'quality_measures', 'overall_rating']
for c in rating_cols:
    stars[c] = pd.to_numeric(stars[c], errors='coerce')

# Quality score = equal-weighted mean of 4 sub-dimensions (1–5 scale)
stars['quality_score'] = stars[['residents_exp','staffing','compliance','quality_measures']].mean(axis=1)

# Parse snapshot string → proper datetime for time-series ordering
def parse_snapshot(s):
    parts = s.split()
    return pd.Timestamp(year=int(parts[1]), month=MONTH_MAP[parts[0]], day=1)

stars['snapshot_date'] = stars['snapshot'].apply(parse_snapshot)

# Care minutes — coerce to numeric (NaN for pre-mandate snapshots or missing sheet)
for col in ['rn_minutes_target', 'rn_minutes_actual', 'total_minutes_target', 'total_minutes_actual']:
    if col in stars.columns:
        stars[col] = pd.to_numeric(stars[col], errors='coerce')
    else:
        stars[col] = pd.NA

# Compliance flags — valid only where both target AND actual are present and target > 0
# Exclude: target=NaN (pre-mandate), actual=NaN (suppressed data), target=0 (data quality issue)
has_rn    = stars['rn_minutes_target'].notna()    & (stars['rn_minutes_target']    > 0) & stars['rn_minutes_actual'].notna()
has_total = stars['total_minutes_target'].notna() & (stars['total_minutes_target'] > 0) & stars['total_minutes_actual'].notna()

stars['rn_compliant']    = (stars['rn_minutes_actual']    >= stars['rn_minutes_target']).where(has_rn)
stars['total_compliant'] = (stars['total_minutes_actual'] >= stars['total_minutes_target']).where(has_total)
stars['fully_compliant'] = (
    (stars['rn_minutes_actual']    >= stars['rn_minutes_target']) &
    (stars['total_minutes_actual'] >= stars['total_minutes_target'])
).where(has_rn & has_total)

# Sanity: post-mandate compliance rates
post = stars[stars['snapshot_date'] >= '2023-10-01']
print('Post-mandate compliance rates (valid data only):')
print(f'  RN minutes compliant   : {post["rn_compliant"].mean()*100:.1f}%')
print(f'  Total minutes compliant: {post["total_compliant"].mean()*100:.1f}%')
print(f'  Fully compliant (both) : {post["fully_compliant"].mean()*100:.1f}%')
pre = stars[stars['snapshot_date'] < '2023-10-01']
print(f'\nPre-mandate rows NaN: {pre["fully_compliant"].isna().all()} (expect True)')
print()
print('Snapshots after fix:')
print(stars[['snapshot','snapshot_date']].drop_duplicates().sort_values('snapshot_date').to_string(index=False))

Post-mandate compliance rates (valid data only):
  RN minutes compliant   : 65.7%
  Total minutes compliant: 56.6%
  Fully compliant (both) : 44.7%

Pre-mandate rows NaN: True (expect True)

Snapshots after fix:
     snapshot snapshot_date
     May 2023    2023-05-01
  August 2023    2023-08-01
December 2023    2023-12-01
February 2024    2024-02-01
     May 2024    2024-05-01
    July 2024    2024-07-01
November 2024    2024-11-01
February 2025    2025-02-01
     May 2025    2025-05-01
  August 2025    2025-08-01
 October 2025    2025-10-01
February 2026    2026-02-01


In [28]:
# =============================================================================
# STEP 3: Join SA3 codes â€” multi-year service list lookup (2025 â†’ 2024 â†’ 2023)
# =============================================================================
# Star ratings files don't include SA3 codes â€” we get them from the service list.
#
# Problem with 2025-only lookup: facilities that closed before 2025 won't appear
# in the 2025 service list, leaving ~7% unmatched. Older snapshots (2023, 2024)
# naturally reference facilities that were still operating then.
#
# Strategy:
#   1. Build SA3 lookups from all three years (2023, 2024, 2025)
#   2. Join on 2025 first (most current SA3 assignment)
#   3. Fill remaining NaN from 2024, then 2023
#
# This is correct because SA3 boundaries don't change between these years â€”
# we're just trying to find which SA3 the facility belongs to, regardless of
# whether it's still operating.

def build_sa3_lookup(year):
    fname = next(f for f in sorted(os.listdir(RAW_SERVICE))
                 if str(year) in f and not f.startswith('~'))
    probe = pd.read_excel(f'{RAW_SERVICE}/{fname}', header=None, nrows=6)
    hdr   = next(i for i, r in probe.iterrows() if 'Service Name' in r.values)
    sl    = pd.read_excel(f'{RAW_SERVICE}/{fname}', header=hdr)
    sa3_col      = next(c for c in sl.columns if 'SA3 Code' in str(c))
    sa3_name_col = next(c for c in sl.columns if 'SA3 Name' in str(c))
    return (
        sl[['Service Name', sa3_col, sa3_name_col]]
        .drop_duplicates('Service Name')
        .rename(columns={sa3_col: 'sa3_code', sa3_name_col: 'sa3_name'})
    )

lookups = {yr: build_sa3_lookup(yr) for yr in [2025, 2024, 2023]}
for yr, lk in lookups.items():
    print(f'Service list {yr}: {len(lk):,} unique facility names')

# Join 2025 first
stars = stars.merge(lookups[2025], on='Service Name', how='left')
print(f'\nAfter 2025 join: {stars["sa3_code"].notna().mean()*100:.1f}% matched')

# Fill from 2024
mask = stars['sa3_code'].isna()
fill24 = stars.loc[mask, ['Service Name']].merge(lookups[2024], on='Service Name', how='left')
stars.loc[mask, 'sa3_code'] = fill24['sa3_code'].values
stars.loc[mask, 'sa3_name'] = fill24['sa3_name'].values
newly_filled_24 = stars['sa3_code'].notna().sum() - (stars.shape[0] - mask.sum())
print(f'After 2024 fill: {stars["sa3_code"].notna().mean()*100:.1f}% matched  (+{mask.sum() - stars["sa3_code"].isna().sum()} rows)')

# Fill from 2023
mask = stars['sa3_code'].isna()
fill23 = stars.loc[mask, ['Service Name']].merge(lookups[2023], on='Service Name', how='left')
stars.loc[mask, 'sa3_code'] = fill23['sa3_code'].values
stars.loc[mask, 'sa3_name'] = fill23['sa3_name'].values
print(f'After 2023 fill: {stars["sa3_code"].notna().mean()*100:.1f}% matched  (+{mask.sum() - stars["sa3_code"].isna().sum()} rows)')

final_unmatched = stars['sa3_code'].isna().sum()
print(f'\nFinal unmatched: {final_unmatched} rows â€” these are facilities not found in any 2023â€“2025 service list')
if final_unmatched > 0:
    print('Sample (likely pre-2023 closures or data entry inconsistencies):')
    print(stars[stars['sa3_code'].isna()]['Service Name'].value_counts().head(10).to_string())

Service list 2025: 5,238 unique facility names
Service list 2024: 5,275 unique facility names
Service list 2023: 5,364 unique facility names

After 2025 join: 93.2% matched
After 2024 fill: 97.7% matched  (+1390 rows)
After 2023 fill: 99.6% matched  (+595 rows)

Final unmatched: 134 rows â€” these are facilities not found in any 2023â€“2025 service list
Sample (likely pre-2023 closures or data entry inconsistencies):
Service Name
Ainsley Nursing Home      3
Aeralife Pennant Hills    3
Melrose Lodge             3
Arcare Aranda             3
Mountain View Lodge       3
Jacaranda Grove           3
Aeralife Illawong         3
Aeralife Botany           3
BaptistCare Minnamurra    3
Aeralife Kingswood        3


In [29]:
# =============================================================================
# STEP 4: Sanity check — quality before vs after mandate by remoteness
# =============================================================================

before = stars[stars['snapshot'] == 'August 2023'].groupby('mmm_code')['quality_score'].mean()
after  = stars[stars['snapshot'] == 'February 2024'].groupby('mmm_code')['quality_score'].mean()

check = pd.DataFrame({'Aug_2023 (pre-mandate)': before, 'Feb_2024 (post-mandate)': after})
check['change'] = check['Feb_2024 (post-mandate)'] - check['Aug_2023 (pre-mandate)']
check['pct_change'] = (check['change'] / check['Aug_2023 (pre-mandate)'] * 100).round(1)
print('Quality change by remoteness (mandate Oct 2023):')
print(check.round(3))
print()
print('>> INSIGHT: If MM6/MM7 show bigger improvements, the mandate may have forced')
print('   underperforming remote facilities to lift their game. But check if facility')
print('   COUNT also changed — some may have closed rather than complied.')


Quality change by remoteness (mandate Oct 2023):
          Aug_2023 (pre-mandate)  Feb_2024 (post-mandate)  change  pct_change
mmm_code                                                                     
MM1                        3.401                    3.499   0.098         2.9
MM2                        3.403                    3.487   0.084         2.5
MM3                        3.361                    3.393   0.032         1.0
MM4                        3.455                    3.592   0.137         4.0
MM5                        3.695                    3.801   0.106         2.9
MM6                        3.643                    3.840   0.197         5.4
MM7                        3.583                    3.906   0.323         9.0

>> INSIGHT: If MM6/MM7 show bigger improvements, the mandate may have forced
   underperforming remote facilities to lift their game. But check if facility
   COUNT also changed — some may have closed rather than complied.


In [30]:
# =============================================================================
# STEP 5: Manual SA3 overrides for facilities missing from service-list lookup
# Source: data/manual/sa3_overrides.csv (built via SAL_2021 → MB_2021 → SA3 chain)
# Recovers ~95% of facilities the service-list join missed.
# =============================================================================

overrides = pd.read_csv('../../data/manual/sa3_overrides.csv')
override_map = {
    r['service_name']: (r['sa3_code'], r['sa3_name'])
    for _, r in overrides.iterrows() if pd.notna(r['sa3_code'])
}

mask = stars['sa3_code'].isna() & stars['Service Name'].isin(override_map.keys())
n_filled = mask.sum()
for name, (code, sname) in override_map.items():
    name_mask = mask & (stars['Service Name'] == name)
    stars.loc[name_mask, 'sa3_code'] = code
    stars.loc[name_mask, 'sa3_name'] = sname
print(f'After manual overrides: {n_filled} rows filled, '
      f'{stars["sa3_code"].notna().mean()*100:.2f}% matched')

# Sentinel for any residual blanks — flag explicitly per data.md Hard Rule #5.
unmapped_mask = stars['sa3_code'].isna()
n_unmapped = unmapped_mask.sum()
stars.loc[unmapped_mask, 'sa3_code'] = -1
stars.loc[unmapped_mask, 'sa3_name'] = 'UNMAPPED'
print(f'After sentinel: {n_unmapped} rows tagged UNMAPPED (sa3_code = -1)')


After manual overrides: 134 rows filled, 100.00% matched
After sentinel: 0 rows tagged UNMAPPED (sa3_code = -1)


In [31]:
# =============================================================================
# STEP 6: Dedup, select columns, write final cleaned star_ratings_by_facility.csv
# =============================================================================

# --- Dedup: some facility-snapshots appear twice in source files.
# Two distinct causes:
#   1. ACQSC revised a rating within the same file (same name+location, different values)
#      → keep first row
#   2. Two genuinely different facilities share the same name (different state or ACPR)
#      → keep BOTH rows (not duplicates)
# Key: snapshot + Service Name + state + acpr_name distinguishes both cases correctly.
before = len(stars)
stars = stars.drop_duplicates(subset=['snapshot', 'Service Name', 'state', 'acpr_name'], keep='first')
print(f'Deduped: {before - len(stars)} rows removed ({len(stars):,} rows remaining)')

# --- Keep only the 24 columns needed for analysis ---
keep_cols = [
    # Identity & time
    'snapshot', 'snapshot_date',
    'Service Name', 'Provider Name', 'Purpose',
    # Geography
    'state', 'acpr_name', 'mmm_code', 'mmm_region',
    # Star ratings
    'overall_rating', 'residents_exp', 'compliance',
    'staffing', 'quality_measures', 'quality_score',
    # Care minutes
    'rn_minutes_target', 'rn_minutes_actual',
    'total_minutes_target', 'total_minutes_actual',
    # Compliance flags
    'rn_compliant', 'total_compliant', 'fully_compliant',
    # SA3 join key
    'sa3_code', 'sa3_name',
]
stars = stars[[c for c in keep_cols if c in stars.columns]]
print(f'Columns kept: {len(stars.columns)}')

stars.to_csv(OUT, index=False)
print(f'✅ Wrote {OUT}')
print(f'   Total rows : {len(stars):,}')
print(f'   Valid SA3  : {(stars["sa3_code"] >= 0).sum():,}')
print(f'   UNMAPPED   : {(stars["sa3_code"] == -1).sum():,}')

Deduped: 113 rows removed (31,177 rows remaining)
Columns kept: 24
✅ Wrote ../../data/clean/star_ratings_by_facility.csv
   Total rows : 31,177
   Valid SA3  : 31,177
   UNMAPPED   : 0


In [32]:
# =============================================================================
# SUMMARY: Output file profile
# =============================================================================
df = pd.read_csv(OUT)

print(f'Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Snapshots      : {df["snapshot_date"].nunique()} ({df["snapshot_date"].min()[:7]} → {df["snapshot_date"].max()[:7]})')
print(f'Unique SA3s    : {df["sa3_code"].nunique()}')
print(f'UNMAPPED (sa3) : {(df["sa3_code"] == -1).sum()}')
print()

nan_notes = {
    'overall_rating'     : 'facility submitted incomplete data that quarter (~5%)',
    'residents_exp'      : 'same — withheld/provisional ratings',
    'staffing'           : 'same — withheld/provisional ratings',
    'compliance'         : 'same — withheld/provisional ratings',
    'quality_measures'   : 'same — withheld/provisional ratings',
    'quality_score'      : 'derived from sub-ratings — NaN when all 4 sub-ratings missing',
    'rn_minutes_target'  : 'pre-mandate snapshots (May/Aug 2023) — no Detailed data sheet',
    'rn_minutes_actual'  : 'same, plus suppressed data for some facilities',
    'total_minutes_target': 'same as rn_minutes_target',
    'total_minutes_actual': 'same as rn_minutes_actual',
    'rn_compliant'       : 'NaN = pre-mandate or missing care minutes data (expected)',
    'total_compliant'    : 'NaN = pre-mandate or missing care minutes data (expected)',
    'fully_compliant'    : 'NaN = pre-mandate or missing care minutes data (expected)',
    'mmm_region'         : 'not present in all snapshot files',
}

nan_counts = df.isnull().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)

print('NaN by column:')
print(f'  {"Column":<25} {"NaN":>7}  {"% ":>5}  Reason')
print(f'  {"-"*25} {"-"*7}  {"-"*5}  {"-"*45}')
for col, n in nan_counts.items():
    pct = n / len(df) * 100
    note = nan_notes.get(col, '— investigate')
    print(f'  {col:<25} {n:>7,}  {pct:>4.1f}%  {note}')

if nan_counts.empty:
    print('  (none)')

Shape          : 31,177 rows × 24 columns
Snapshots      : 12 (2023-05 → 2026-02)
Unique SA3s    : 323
UNMAPPED (sa3) : 0

NaN by column:
  Column                        NaN     %   Reason
  ------------------------- -------  -----  ---------------------------------------------
  total_compliant             6,366  20.4%  NaN = pre-mandate or missing care minutes data (expected)
  fully_compliant             6,366  20.4%  NaN = pre-mandate or missing care minutes data (expected)
  rn_compliant                6,366  20.4%  NaN = pre-mandate or missing care minutes data (expected)
  total_minutes_actual        5,890  18.9%  same as rn_minutes_actual
  total_minutes_target        5,878  18.9%  same as rn_minutes_target
  rn_minutes_target           5,878  18.9%  pre-mandate snapshots (May/Aug 2023) — no Detailed data sheet
  rn_minutes_actual           5,774  18.5%  same, plus suppressed data for some facilities
  overall_rating              2,138   6.9%  facility submitted incomplete data